In [2]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.ops import unary_union

from pathlib import Path
from tqdm.auto import tqdm

datasets = Path("/nas/cee-water/cjgleason/data")
proj_root = Path("/nas/cee-water/cjgleason/ted/swot-ml")

save_dir = proj_root / 'data' / 'multigraph'
metadata_dir = save_dir / 'metadata'
attributes_dir = save_dir / 'attributes'

matchups = gpd.read_parquet(metadata_dir / "All_MERIT_matchups.parquet").set_index('comid')
matchups.index = matchups.index.astype(str)
footprint_geom = unary_union(matchups.geometry.envelope)

# Read in hydroatlas
gdb_path = datasets / "HydroSHEDS" / "HydroATLAS" / "BasinATLAS_v10.gdb"
layer_name = "BasinATLAS_v10_lev09"

hydroatlas = gpd.read_file(gdb_path, layer=layer_name, engine="pyogrio", use_arrow=True)
hydroatlas.replace({'wet_cl_smj':-9999}, 13, inplace=True)
hydroatlas = hydroatlas.set_index('HYBAS_ID')

/nas/cee-water/cjgleason/ted/swot-ml/.venv/lib/python3.11/site-packages/pyogrio/raw.py:337: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  table = reader.read_all()


In [3]:
hydroatlas_filt = hydroatlas[hydroatlas.intersects(footprint_geom)]
hydroatlas_filt

,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,PFAF_ID,ENDO,COAST,...,hft_ix_s09,hft_ix_u09,gad_id_smj,gdp_ud_sav,gdp_ud_ssu,gdp_ud_usu,hdi_ix_sav,Shape_Length,Shape_Area,geometry
HYBAS_ID,,,,,,,,,,,,,,,,,,,,,
2.090471e+09,2.090471e+09,2.090008e+09,2.090008e+09,2742.9,2742.9,587.8,5356.2,227999910,0,0,...,187,177,86,47765,5.371887e+09,3.498405e+10,941,1.658522,0.071262,"MULTIPOLYGON (((9.98333 47.89583, 9.97696 47.8..."
2.090505e+09,2.090502e+09,2.090008e+09,2.090008e+09,2756.3,2756.3,729.6,729.6,227980960,0,0,...,150,150,15,45967,7.845448e+08,7.845449e+08,896,1.517652,0.086590,"MULTIPOLYGON (((10.24583 47.15417, 10.24673 47..."
2.090474e+09,2.090471e+09,2.090008e+09,2.090008e+09,2762.3,2762.3,509.7,509.8,227999920,0,0,...,190,190,86,47765,3.727289e+09,3.727288e+09,941,1.376472,0.061704,"MULTIPOLYGON (((9.7875 47.8875, 9.7875 47.9, 9..."
2.090483e+09,2.090480e+09,2.090008e+09,2.090008e+09,2798.9,2798.9,787.5,1358.3,227999805,0,0,...,181,173,86,50091,7.867913e+09,1.026224e+10,930,2.216932,0.094637,"MULTIPOLYGON (((10.2125 47.72917, 10.21165 47...."
2.090483e+09,2.090480e+09,2.090008e+09,2.090008e+09,2798.9,2798.9,364.9,365.0,227999804,0,0,...,163,163,86,47996,1.999762e+09,1.999761e+09,940,1.530200,0.043970,"MULTIPOLYGON (((9.82917 47.84167, 9.82286 47.8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8.090193e+09,8.090194e+09,8.090042e+09,8.090042e+09,370.6,370.6,137.6,137.6,852049802,0,0,...,0,0,41,64789,0.000000e+00,0.000000e+00,832,0.977953,0.025887,"MULTIPOLYGON (((-69.725 64.47917, -69.725 64.4..."
8.090194e+09,8.090195e+09,8.090042e+09,8.090042e+09,385.1,385.1,136.9,137.0,852049904,0,0,...,0,0,41,64789,0.000000e+00,0.000000e+00,832,1.135298,0.025677,"MULTIPOLYGON (((-69.42917 64.42917, -69.46076 ..."
8.090194e+09,8.090195e+09,8.090042e+09,8.090042e+09,385.2,385.2,281.2,1533.0,852049905,0,0,...,0,0,41,64789,0.000000e+00,0.000000e+00,832,1.368882,0.052953,"MULTIPOLYGON (((-69.26667 64.65, -69.26575 64...."


In [4]:
outlets = matchups.copy()
outlets.geometry = gpd.GeoSeries.from_wkt(outlets['outlet']).set_crs('EPSG:4326')
outlets_buff = outlets.to_crs("EPSG:6933")
outlets_buff.geometry = outlets_buff.buffer(2500)
outlets_buff

hybas_pairs = gpd.sjoin(
    outlets_buff,
    hydroatlas_filt.to_crs("EPSG:6933"),
    how='left',
)

hybas_area = hybas_pairs['UP_AREA']
merit_area = hybas_pairs['total_area']
hybas_pairs['hybas_area_diff'] = (hybas_area - merit_area) / ((hybas_area + merit_area)/2)

def get_best_hybas(grp):
     # Pick best match by area.
    idx_min = grp['hybas_area_diff'].abs().idxmin()
    return grp.loc[idx_min]

all_attributes = hybas_pairs.reset_index().groupby('comid').apply(get_best_hybas, include_groups=False)

In [10]:
all_attributes['s2m_values']

comid
21000001                          []
21000012                          []
21000019                          []
21000021                          []
21000022                          []
                            ...     
USGS-410401112134801              []
USGS-50035000                     []
USGS-50037000                     []
USGS-50055000           ['76004138']
USGS-50059050                     []
Name: s2m_values, Length: 36622, dtype: object

In [20]:
gauge_cols = [
    'outlet',
    'outlet_id',
    'total_area',
    'unitarea',
    'reservoir',
    'custom',
    'HYBAS_ID',
    'hybas_area_diff',
    'reach_id',
    'sword_area',
    'sword_distance',
    'lake_reach_ids',
    'lake_pld_ids',
    's2m_values',
    'name',
    'area',
    'latitude',
    'longitude',
    'min_date',
    'max_date',
    'min_discharge',
    'max_discharge',
    'mean_discharge',
    'count_discharge',
    'provider',
]
gauge_matches = all_attributes[gauge_cols].dropna(subset='mean_discharge')
gauge_matches.index.name = 'subbasin'

gauge_matches.to_parquet(metadata_dir / 'gauge_matches.parquet')

In [21]:
test = pd.read_parquet(metadata_dir / 'gauge_matches.parquet')

test

,outlet,outlet_id,total_area,unitarea,reservoir,custom,HYBAS_ID,hybas_area_diff,reach_id,sword_area,...,area,latitude,longitude,min_date,max_date,min_discharge,max_discharge,mean_discharge,count_discharge,provider
subbasin,,,,,,,,,,,,,,,,,,,,,
ABOM-100288010,POINT (115.8692 -32.6175),ABOM-100288010,7294.5,302.9,False,True,5.090574e+09,-0.026546,NaN,NaN,...,NaN,-32.633517,115.878081,1991-10-24,2025-09-07,0.019000,289.706000,7.570296,12049.0,abom
ABOM-101491010,POINT (115.9517 -32.3392),ABOM-98416010,832.3,568.5,False,True,5.090570e+09,0.043143,NaN,NaN,...,NaN,-32.337197,115.885163,1998-06-19,2025-09-07,0.001000,21.856000,0.441056,9546.0,abom
ABOM-101896010,POINT (116.6083 -32.8142),ABOM-100288010,350.8,350.7,False,True,5.090576e+09,0.129815,NaN,NaN,...,NaN,-32.826625,116.618006,2008-03-31,2025-09-07,0.001000,55.781000,0.572767,5100.0,abom
ABOM-102903010,POINT (116.4558 -33.0108),ABOM-100288010,1386.5,246.6,False,True,5.090578e+09,0.035562,NaN,NaN,...,NaN,-32.994466,116.428563,1966-06-19,2025-09-07,0.001000,227.972000,1.997985,21376.0,abom
ABOM-102984010,POINT (116.3933 -32.8542),ABOM-100288010,4037.4,909.0,False,True,5.090577e+09,-0.009306,NaN,NaN,...,NaN,-32.865404,116.397904,1966-06-16,2025-09-07,0.001000,244.823000,3.222303,21077.0,abom
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
USGS-410401112134801,POINT (-112.1875 41.0625),USGS-410401112134801,9900.3,296.5,False,True,7.090513e+09,-0.002093,NaN,NaN,...,NaN,41.066564,-112.230256,2003-10-01,2025-09-08,0.006230,45.873291,11.263905,7165.0,usgs
USGS-50035000,POINT (-66.4592 18.3217),USGS-50037000,345.7,345.7,False,True,7.090073e+09,0.109379,NaN,NaN,...,347.056682,18.322438,-66.459568,1950-01-01,2025-09-08,0.240693,1857.585136,7.406389,25691.0,usgs
USGS-50037000,POINT (-66.5 18.3983),USGS-50037000,429.1,83.4,False,True,7.090073e+09,0.353415,NaN,NaN,...,409.216087,18.397448,-66.496560,2019-06-13,2025-09-08,1.265763,911.802460,11.017469,2247.0,usgs


In [26]:
test.hybas_area_diff.isna().mean()

0.0

In [22]:
list(test)

['outlet',
 'outlet_id',
 'total_area',
 'unitarea',
 'reservoir',
 'custom',
 'HYBAS_ID',
 'hybas_area_diff',
 'reach_id',
 'sword_area',
 'sword_distance',
 'lake_reach_ids',
 'lake_pld_ids',
 's2m_values',
 'name',
 'area',
 'latitude',
 'longitude',
 'min_date',
 'max_date',
 'min_discharge',
 'max_discharge',
 'mean_discharge',
 'count_discharge',
 'provider']

In [10]:
hybas_properties = pd.read_csv(attributes_dir / "properties.txt", header=None)[0].to_list()
matchup_props = ['outlet_id', 'unitarea','reservoir','total_area']
all_props = matchup_props + hybas_properties + ['hybas_area_diff']

In [25]:
attributes = (
    all_attributes[all_props]
    .reset_index()
    .rename(columns={'comid':'subbasin', 'outlet_id': 'basin'})
    .astype({'subbasin':str, 'basin': str})
    .set_index(['basin', 'subbasin'])
)
attributes

unitarea  reservoir  total_area  \
basin                subbasin                                                
EAUF-V7200010        21000001                 152.9      False       398.5   
                     21000012                 194.6      False       324.8   
                     21000019                 242.5      False      5173.0   
                     21000021                   8.4      False      4758.7   
                     21000022                  68.9      False      4506.6   
...                                             ...        ...         ...   
USGS-410401112134801 USGS-410401112134801     296.5      False      9900.3   
USGS-50037000        USGS-50035000            345.7      False       345.7   
                     USGS-50037000             83.4      False       429.1   
USGS-50059050        USGS-50055000            232.3      False       232.4   
                     USGS-50059050            308.9      False       541.3   

                                           dis_m3_pyr  dis_m3_pmn  dis_m3_pmx  \
basin                subbasin                                                   
EAUF-V7200010        21000001                6.031000    2.077000    9.826000   
                     21000012                6.031000    2.077000    9.826000   
                     21000019               76.288002   25.957001  125.834999   
                     21000021               69.873001   23.823000  115.067001   
                     21000022               69.873001   23.823000  115.067001   
...                                               ...         ...         ...   
USGS-410401112134801 USGS-410401112134801   55.528999   22.336000  141.119995   
USGS-50037000        USGS-50035000           0.020000    0.009000    0.036000   
                     USGS-50037000          10.798000    4.773000   20.870001   
USGS-50059050        USGS-50055000          26.511999   13.490000   38.925999   
                     USGS-50059050           0.782000    0.323000    1.420000   

                                           run_mm_syr  inu_pc_smn  inu_pc_smx  \
basin                subbasin                                                   
EAUF-V7200010        21000001                     422           0           0   
                     21000012                     422           0           0   
                     21000019                     491           7           7   
                     21000021                     500           0           0   
                     21000022                     500           0           0   
...                                               ...         ...         ...   
USGS-410401112134801 USGS-410401112134801         148          70          83   
USGS-50037000        USGS-50035000                564          36          40   
                     USGS-50037000                561          21          39   
USGS-50059050        USGS-50055000                699          25          59   
                     USGS-50059050                628          55          62   

                                           inu_pc_slt  ...  urb_pc_sse  \
basin                subbasin                          ...               
EAUF-V7200010        21000001                       0  ...           0   
                     21000012                       0  ...           0   
                     21000019                      19  ...           0   
                     21000021                      11  ...           0   
                     21000022                      11  ...           0   
...                                               ...  ...         ...   
USGS-410401112134801 USGS-410401112134801          86  ...          16   
USGS-50037000        USGS-50035000                 41  ...          46   
                     USGS-50037000                 41  ...          17   
USGS-50059050        USGS-50055000                 59  ...          51   
                     USGS-500590

In [33]:
attributes.to_parquet(attributes_dir / f"static.parquet")

In [34]:
test = pd.read_parquet(attributes_dir / "static.parquet")

In [35]:
test

unitarea  reservoir  total_area  \
basin                subbasin                                                
EAUF-V7200010        21000001                 152.9      False       398.5   
                     21000012                 194.6      False       324.8   
                     21000019                 242.5      False      5173.0   
                     21000021                   8.4      False      4758.7   
                     21000022                  68.9      False      4506.6   
...                                             ...        ...         ...   
USGS-410401112134801 USGS-410401112134801     296.5      False      9900.3   
USGS-50037000        USGS-50035000            345.7      False       345.7   
                     USGS-50037000             83.4      False       429.1   
USGS-50059050        USGS-50055000            232.3      False       232.4   
                     USGS-50059050            308.9      False       541.3   

                                           dis_m3_pyr  dis_m3_pmn  dis_m3_pmx  \
basin                subbasin                                                   
EAUF-V7200010        21000001                6.031000    2.077000    9.826000   
                     21000012                6.031000    2.077000    9.826000   
                     21000019               76.288002   25.957001  125.834999   
                     21000021               69.873001   23.823000  115.067001   
                     21000022               69.873001   23.823000  115.067001   
...                                               ...         ...         ...   
USGS-410401112134801 USGS-410401112134801   55.528999   22.336000  141.119995   
USGS-50037000        USGS-50035000           0.020000    0.009000    0.036000   
                     USGS-50037000          10.798000    4.773000   20.870001   
USGS-50059050        USGS-50055000          26.511999   13.490000   38.925999   
                     USGS-50059050           0.782000    0.323000    1.420000   

                                           run_mm_syr  inu_pc_smn  inu_pc_smx  \
basin                subbasin                                                   
EAUF-V7200010        21000001                     422           0           0   
                     21000012                     422           0           0   
                     21000019                     491           7           7   
                     21000021                     500           0           0   
                     21000022                     500           0           0   
...                                               ...         ...         ...   
USGS-410401112134801 USGS-410401112134801         148          70          83   
USGS-50037000        USGS-50035000                564          36          40   
                     USGS-50037000                561          21          39   
USGS-50059050        USGS-50055000                699          25          59   
                     USGS-50059050                628          55          62   

                                           inu_pc_slt  ...  urb_pc_sse  \
basin                subbasin                          ...               
EAUF-V7200010        21000001                       0  ...           0   
                     21000012                       0  ...           0   
                     21000019                      19  ...           0   
                     21000021                      11  ...           0   
                     21000022                      11  ...           0   
...                                               ...  ...         ...   
USGS-410401112134801 USGS-410401112134801          86  ...          16   
USGS-50037000        USGS-50035000                 41  ...          46   
                     USGS-50037000                 41  ...          17   
USGS-50059050        USGS-50055000                 59  ...          51   
                     USGS-500590